In [ ]:
#pip install torch

In [ ]:
import os
os.environ["USE_TF"] = "0"   # Prevent TensorFlow DLL crash on Windows/Py3.12

import re
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA

from transformers import AutoTokenizer, AutoModel
import torch

# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION  — update paths before running
# ═══════════════════════════════════════════════════════════════════
NOTE_DIR    = r"C:\Users\anura\OneDrive\Desktop\data"
LABELS_PATH = r"C:\Users\anura\OneDrive\Desktop\data\labels.csv"
OUTPUT_DIR  = r"C:\Users\anura\OneDrive\Desktop\data\outputs"
CACHE_DIR   = os.path.join(OUTPUT_DIR, "pipeline_cache")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)

# ── NLP hyper-parameters (paper Section 6.2 / 6.3) ───────────────
# Paper: 500-term TF-IDF vocab; SVD retains 80% variance;
#        BioBERT 768-dim mean-pool; PCA retains 90% variance.
TFIDF_VOCAB_SIZE    = 500
SVD_VARIANCE_TARGET = 0.80
PCA_VARIANCE_TARGET = 0.90
BERT_MODEL_NAME     = "dmis-lab/biobert-base-cased-v1.2"
BERT_MAX_LEN        = 512
RANDOM_STATE        = 42


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  DEVICE CHECK
# ═══════════════════════════════════════════════════════════════════

def check_and_report_device():
    if torch.cuda.is_available():
        device     = torch.device("cuda")
        gpu_name   = torch.cuda.get_device_name(0)
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        bert_batch = 32
        print(f"  ✅  GPU detected : {gpu_name}  ({gpu_mem_gb:.1f} GB)")
        print(f"      BERT batch size : {bert_batch}")
    else:
        device     = torch.device("cpu")
        bert_batch = 8
        print(f"  ⚠️   No GPU — running on CPU")
        print(f"      BERT batch size : {bert_batch} (CPU-safe)")
        print(f"      Estimated BioBERT time : 4–8 hours")
        print(f"      TIP: Use Google Colab (free GPU) for 50× speedup.")
    return device, bert_batch


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STAGE 1 — LOAD LABELS + FILTER NOTES
#  Paper Section 6.1:
#    • Link notes to 2,307 ICU patients via hadm_id
#    • Discharge : keep highest note_seq (most complete summary)
#    • Radiology : keep lowest  note_seq (earliest report)
#    • Result: 1,618 discharge (70.1%) + 1,639 radiology (71.0%)
# ═══════════════════════════════════════════════════════════════════

def load_labels():
    """
    Load the minimal labels file: hadm_id + mortality
    """

    struct = pd.read_csv(r"C:\Users\anura\Downloads\mimic_final_dataset1.csv")
    struct[["hadm_id", "mortality"]].to_csv("labels.csv", index=False)

    print("=" * 60)
    print("LOADING LABELS")
    print("=" * 60)
    labels = pd.read_csv(LABELS_PATH, low_memory=False)
    labels["hadm_id"] = labels["hadm_id"].astype(int)
    print(f"  Total labelled admissions : {len(labels):,}")
    print(f"  Mortality rate            : {labels['mortality'].mean()*100:.1f}%")
    print(f"  Deaths   : {labels['mortality'].sum():,}  |  "
          f"Survivors : {(labels['mortality']==0).sum():,}")
    return labels


def load_and_filter_notes(cohort_hadm_ids):
    """
    Load discharge and radiology notes, filter to the ICU cohort,
    and select ONE note per patient per modality (paper Section 6.1).

    Discharge : highest note_seq → most complete final summary
    Radiology : lowest  note_seq → earliest report (temporal relevance)

    Each result is cached as .parquet after the first load.
    """
    disch_cache = os.path.join(CACHE_DIR, "discharge_filtered.parquet")
    radio_cache = os.path.join(CACHE_DIR, "radiology_filtered.parquet")

    print("\n" + "=" * 60)
    print("STAGE 1 — Load and filter notes  (paper Section 6.1)")
    print("=" * 60)

    # ── Discharge ────────────────────────────────────────────────
    if os.path.exists(disch_cache):
        print("\n  [CACHE HIT] discharge notes")
        discharge_df = pd.read_parquet(disch_cache)
    else:
        print("\n  Loading discharge.csv in chunks...")
        usecols = ["note_id", "subject_id", "hadm_id",
                   "note_type", "note_seq", "charttime", "text"]
        chunks = []
        for chunk in tqdm(
            pd.read_csv(os.path.join(NOTE_DIR, "discharge.csv"),
                        usecols=usecols, chunksize=50_000, low_memory=False),
            desc="  discharge chunks"
        ):
            filtered = chunk[chunk["hadm_id"].isin(cohort_hadm_ids)]
            if not filtered.empty:
                chunks.append(filtered)
        # keep highest note_seq per patient (most complete summary)
        discharge_df = (
            pd.concat(chunks, ignore_index=True)
            .sort_values("note_seq", ascending=False)
            .groupby("hadm_id", as_index=False).first()
        )
        discharge_df.to_parquet(disch_cache, index=False)

    print(f"  Discharge notes : {len(discharge_df):,}  "
          f"({100*len(discharge_df)/len(cohort_hadm_ids):.1f}% coverage)")

    # ── Radiology ────────────────────────────────────────────────
    if os.path.exists(radio_cache):
        print("\n  [CACHE HIT] radiology notes")
        radiology_df = pd.read_parquet(radio_cache)
    else:
        print("\n  Loading radiology.csv in chunks...")
        usecols = ["note_id", "subject_id", "hadm_id",
                   "note_type", "note_seq", "charttime", "text"]
        chunks = []
        for chunk in tqdm(
            pd.read_csv(os.path.join(NOTE_DIR, "radiology.csv"),
                        usecols=usecols, chunksize=50_000, low_memory=False),
            desc="  radiology chunks"
        ):
            filtered = chunk[chunk["hadm_id"].isin(cohort_hadm_ids)]
            if not filtered.empty:
                chunks.append(filtered)
        # keep lowest note_seq per patient (earliest report)
        radiology_df = (
            pd.concat(chunks, ignore_index=True)
            .sort_values("note_seq", ascending=True)
            .groupby("hadm_id", as_index=False).first()
        )
        radiology_df.to_parquet(radio_cache, index=False)

    print(f"  Radiology notes : {len(radiology_df):,}  "
          f"({100*len(radiology_df)/len(cohort_hadm_ids):.1f}% coverage)")

    return discharge_df, radiology_df


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STAGE 2 — TEXT NORMALIZATION
#  Paper Section 6.2:
#    1. Replace MIMIC de-id underscores (___) with space
#    2. Lowercase
#    3. Remove digits
#    4. Remove punctuation / special characters
#    5. Collapse whitespace
# ═══════════════════════════════════════════════════════════════════

def clean_text(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    text = re.sub(r"_{2,}", " ", text)   # de-id placeholders
    text = text.lower()
    text = re.sub(r"\d+", " ", text)     # remove digits
    text = re.sub(r"[^a-z\s]", " ", text) # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_notes(discharge_df, radiology_df):
    print("\n" + "=" * 60)
    print("STAGE 2 — Text normalization  (paper Section 6.2)")
    print("=" * 60)
    discharge_df = discharge_df.copy()
    radiology_df = radiology_df.copy()
    discharge_df["clean_text"] = discharge_df["text"].apply(clean_text)
    radiology_df["clean_text"] = radiology_df["text"].apply(clean_text)
    discharge_df["text_valid"] = discharge_df["clean_text"].str.len() > 10
    radiology_df["text_valid"] = radiology_df["clean_text"].str.len() > 10
    print(f"  Discharge valid texts : {discharge_df['text_valid'].sum():,} "
          f"({discharge_df['text_valid'].mean()*100:.1f}%)")
    print(f"  Radiology valid texts : {radiology_df['text_valid'].sum():,} "
          f"({radiology_df['text_valid'].mean()*100:.1f}%)")
    return discharge_df, radiology_df


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STAGE 3A — TF-IDF + TRUNCATEDSVD
#  Paper Section 6.2 / 6.3:
#    • 500-term vocabulary (built separately for each note type)
#    • Sparse document-term matrix → TruncatedSVD
#    • Retain components explaining 80% of variance
#    • Paper result: 136 components (discharge), 198 (radiology)
#
#  No train/test split needed here — we are extracting features
#  for ALL patients to produce the final feature dataset.
# ═══════════════════════════════════════════════════════════════════

def tfidf_svd_pipeline(texts_all, hadm_ids_all, prefix,
                        vocab_size=TFIDF_VOCAB_SIZE,
                        variance_target=SVD_VARIANCE_TARGET):
    """
    Fit TF-IDF + TruncatedSVD on ALL texts, transform all texts.
    Cached as .npy after first run.
    """
    cache_path = os.path.join(CACHE_DIR, f"{prefix}_tfidf_svd.npy")
    meta_path  = os.path.join(CACHE_DIR, f"{prefix}_tfidf_svd_meta.npz")

    print(f"\n  [{prefix}] TF-IDF + SVD pipeline  (paper Sections 6.2–6.3)")

    if os.path.exists(cache_path) and os.path.exists(meta_path):
        print(f"  [{prefix}] [CACHE HIT]")
        reduced   = np.load(cache_path)
        meta      = np.load(meta_path, allow_pickle=True)
        n_keep    = int(meta["n_keep"])
        col_names = [f"{prefix}_tfidf_svd_{i}" for i in range(n_keep)]
        feat_df   = pd.DataFrame(reduced, columns=col_names)
        feat_df["hadm_id"] = hadm_ids_all
        print(f"  [{prefix}] Loaded {reduced.shape}  n_keep={n_keep}")
        return feat_df, n_keep

    # ── TF-IDF ───────────────────────────────────────────────────
    # paper: lowercased, punctuation/digits/stopwords removed (done in Stage 2)
    # 500-term vocab, bigrams allowed, sublinear TF scaling
    tfidf = TfidfVectorizer(
        max_features=vocab_size,
        stop_words="english",
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        ngram_range=(1, 2)
    )
    tfidf_matrix = tfidf.fit_transform(texts_all)   # fit + transform all
    print(f"  [{prefix}] TF-IDF matrix : {tfidf_matrix.shape}")

    # ── TruncatedSVD ─────────────────────────────────────────────
    max_components = min(tfidf_matrix.shape[1] - 1,
                         tfidf_matrix.shape[0] - 1, 300)
    svd_full = TruncatedSVD(n_components=max_components,
                             random_state=RANDOM_STATE)
    svd_full.fit(tfidf_matrix)
    cumvar = np.cumsum(svd_full.explained_variance_ratio_)
    n_keep = min(int(np.searchsorted(cumvar, variance_target)) + 1,
                 max_components)
    print(f"  [{prefix}] SVD components for {variance_target*100:.0f}% variance : {n_keep}")

    reduced = svd_full.transform(tfidf_matrix)[:, :n_keep]

    # ── Variance plot ─────────────────────────────────────────────
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(cumvar) + 1), cumvar,
             linewidth=1.5, color="#378ADD")
    plt.axhline(variance_target, color="#E24B4A", linestyle="--",
                label=f"Target {variance_target*100:.0f}%")
    plt.axvline(n_keep, color="#1D9E75", linestyle="--",
                label=f"{n_keep} components")
    plt.xlabel("SVD components")
    plt.ylabel("Cumulative variance")
    plt.title(f"{prefix} TF-IDF — Cumulative explained variance")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,
                f"{prefix}_tfidf_svd_variance.png"), dpi=150)
    plt.show()

    np.save(cache_path, reduced)
    np.savez(meta_path, n_keep=np.array(n_keep))

    col_names = [f"{prefix}_tfidf_svd_{i}" for i in range(n_keep)]
    feat_df   = pd.DataFrame(reduced, columns=col_names)
    feat_df["hadm_id"] = hadm_ids_all
    print(f"  [{prefix}] TF-IDF+SVD output shape : {feat_df.shape}")
    return feat_df, n_keep


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STAGE 3B — BIOBERT EMBEDDINGS + PCA
#  Paper Section 6.2 / 6.3:
#    • BioBERT (Alsentzer et al. 2019) — pretrained on PubMed + PMC
#    • Mean-pool final hidden layer → 768-dim dense vector per note
#    • PCA retains 90% of variance
#    • Paper result: 113 components (discharge), 115 (radiology)
#
#  No train/test split — encode and reduce ALL patient notes.
# ═══════════════════════════════════════════════════════════════════

def get_bert_embeddings(texts, tokenizer, model, device,
                         batch_size, desc="BioBERT"):
    """
    Mean-pool BioBERT final hidden states over non-padding tokens.
    (paper Section 6.2 — 768-dim dense vector per document)
    """
    model = model.to(device)
    model.eval()
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"  {desc}"):
        batch   = texts[i: i + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=BERT_MAX_LEN, return_tensors="pt")
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
        token_emb = output.last_hidden_state
        attn_mask = encoded["attention_mask"]
        mask_exp  = attn_mask.unsqueeze(-1).float()
        sum_emb   = (token_emb * mask_exp).sum(dim=1)
        sum_mask  = mask_exp.sum(dim=1).clamp(min=1e-9)
        mean_emb  = (sum_emb / sum_mask).cpu().numpy()
        all_embeddings.append(mean_emb)
    return np.vstack(all_embeddings)


def bert_pca_pipeline(texts_all, hadm_ids_all, prefix,
                       tokenizer, bert_model, device, bert_batch_size,
                       variance_target=PCA_VARIANCE_TARGET):
    """
    Encode all texts with BioBERT, then reduce with PCA.
    Embeddings and PCA output cached as .npy after first run.
    """
    emb_cache      = os.path.join(CACHE_DIR, f"{prefix}_bert_emb.npy")
    pca_feat_cache = os.path.join(CACHE_DIR, f"{prefix}_bert_pca.npy")
    pca_meta_cache = os.path.join(CACHE_DIR, f"{prefix}_bert_pca_meta.npz")

    print(f"\n  [{prefix}] BioBERT + PCA pipeline  (paper Sections 6.2–6.3)")

    # ── Full PCA output cached ────────────────────────────────────
    if os.path.exists(pca_feat_cache) and os.path.exists(pca_meta_cache):
        print(f"  [{prefix}] [CACHE HIT]")
        reduced   = np.load(pca_feat_cache)
        meta      = np.load(pca_meta_cache, allow_pickle=True)
        n_keep    = int(meta["n_keep"])
        col_names = [f"{prefix}_bert_pca_{i}" for i in range(n_keep)]
        feat_df   = pd.DataFrame(reduced, columns=col_names)
        feat_df["hadm_id"] = hadm_ids_all
        return feat_df, n_keep

    # ── BioBERT encoding ─────────────────────────────────────────
    if os.path.exists(emb_cache):
        print(f"  [{prefix}] [CACHE HIT] embeddings")
        embeddings = np.load(emb_cache)
    else:
        t0 = time.time()
        print(f"  [{prefix}] Encoding {len(texts_all):,} texts with BioBERT...")
        embeddings = get_bert_embeddings(
            texts_all, tokenizer, bert_model, device, bert_batch_size,
            desc=f"{prefix} BioBERT"
        )
        np.save(emb_cache, embeddings)
        print(f"  [{prefix}] Encoding done in {(time.time()-t0)/60:.1f} min")

    print(f"  [{prefix}] Embedding matrix : {embeddings.shape}")

    # ── PCA ──────────────────────────────────────────────────────
    max_comp = min(embeddings.shape[0] - 1, embeddings.shape[1])
    pca_full = PCA(n_components=max_comp, random_state=RANDOM_STATE)
    pca_full.fit(embeddings)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_keep = min(int(np.searchsorted(cumvar, variance_target)) + 1, max_comp)
    print(f"  [{prefix}] PCA components for {variance_target*100:.0f}% variance : {n_keep}")

    # ── Variance plot ─────────────────────────────────────────────
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(cumvar) + 1), cumvar,
             linewidth=1.5, color="#1D9E75")
    plt.axhline(variance_target, color="#E24B4A", linestyle="--",
                label=f"Target {variance_target*100:.0f}%")
    plt.axvline(n_keep, color="#378ADD", linestyle="--",
                label=f"{n_keep} components")
    plt.xlabel("PCA components")
    plt.ylabel("Cumulative variance")
    plt.title(f"{prefix} BioBERT — Cumulative explained variance")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,
                f"{prefix}_bert_pca_variance.png"), dpi=150)
    plt.show()

    pca     = PCA(n_components=n_keep, random_state=RANDOM_STATE)
    reduced = pca.fit_transform(embeddings)
    np.save(pca_feat_cache, reduced)
    np.savez(pca_meta_cache, n_keep=np.array(n_keep))

    col_names = [f"{prefix}_bert_pca_{i}" for i in range(n_keep)]
    feat_df   = pd.DataFrame(reduced, columns=col_names)
    feat_df["hadm_id"] = hadm_ids_all
    print(f"  [{prefix}] BioBERT+PCA output shape : {feat_df.shape}")
    return feat_df, n_keep


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STAGE 4 — ASSEMBLE TEXT FEATURES + ZERO-FILL MISSING
#  Paper Section 6.4:
#    • Merge all feature DataFrames with labels via hadm_id
#    • Zero-fill missing embeddings (absence of note → zero vector)
#      Valid because SVD/PCA outputs are approximately zero-centred
#    • Add binary note_present flags so the model can distinguish
#      "no document" from a genuine near-zero embedding
# ═══════════════════════════════════════════════════════════════════

def assemble_text_features(labels_df, disch_tfidf, radio_tfidf,
                            disch_bert, radio_bert):
    print("\n" + "=" * 60)
    print("STAGE 4 — Assemble text-only dataset  (paper Section 6.4)")
    print("=" * 60)

    merged = labels_df[["hadm_id", "mortality"]].copy()

    for feat_df, name in [
        (disch_tfidf, "disch_tfidf"),
        (radio_tfidf, "radio_tfidf"),
        (disch_bert,  "disch_bert"),
        (radio_bert,  "radio_bert"),
    ]:
        has_note = set(feat_df["hadm_id"])
        flag_col = name.split("_")[0] + "_note_present"
        if flag_col not in merged.columns:
            merged[flag_col] = merged["hadm_id"].isin(has_note).astype(int)
        merged = merged.merge(feat_df, on="hadm_id", how="left")

    embed_cols = [c for c in merged.columns
                  if any(x in c for x in ["tfidf_svd", "bert_pca"])]
    n_missing  = merged[embed_cols].isna().sum().sum()
    merged[embed_cols] = merged[embed_cols].fillna(0)   # zero-imputation

    print(f"  Total text feature columns  : {len(embed_cols)}")
    print(f"  Missing cells → zero-filled : {n_missing:,}")
    print(f"  Final dataset shape         : {merged.shape}")
    for flag in ["disch_note_present", "radio_note_present"]:
        if flag in merged.columns:
            n   = merged[flag].sum()
            pct = 100 * merged[flag].mean()
            print(f"  {flag} : {n:,} ({pct:.1f}% coverage)")


    STRUCT_PATH = r"C:\Users\anura\OneDrive\Desktop\data\mimic_final_dataset1"
    struct = pd.read_csv(STRUCT_PATH, low_memory=False)
    struct["hadm_id"] = struct["hadm_id"].astype(int)
    multimodal = struct.merge(merged, on="hadm_id", how="left")
    embed_cols = [c for c in multimodal.columns
    if any(x in c for x in ["tfidf_svd", "bert_pca"])]
    multimodal[embed_cols] = multimodal[embed_cols].fillna(0)
    print(f"  Multimodal dataset shape : {multimodal.shape}")
    return multimodal



In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  MAIN — run all stages in sequence
#  No train/test split: we are producing a complete feature dataset
#  for all 2,307 patients (paper Section 6.1–6.4).
#  The split is only needed when modeling (paper Sections 7–8).
# ═══════════════════════════════════════════════════════════════════

def main():
    pipeline_start = time.time()

    print("\n" + "=" * 60)
    print("TEXT-ONLY FEATURE EXTRACTION PIPELINE")
    print("Replicating paper Sections 6.1 – 6.4")
    print("=" * 60)

    # ── Device ────────────────────────────────────────────────────
    device, bert_batch_size = check_and_report_device()

    # ── Load labels ───────────────────────────────────────────────
    labels          = load_labels()
    cohort_hadm_ids = set(labels["hadm_id"].astype(int))

    # ── Stage 1: Load + filter notes ─────────────────────────────
    discharge_df, radiology_df = load_and_filter_notes(cohort_hadm_ids)

    # ── Stage 2: Normalise text ───────────────────────────────────
    discharge_df, radiology_df = normalize_notes(discharge_df, radiology_df)

    # Helper: extract clean text list + hadm_id list for a notes df
    def get_texts(notes_df):
        return (notes_df["clean_text"].tolist(),
                notes_df["hadm_id"].tolist())

    d_texts, d_hadm = get_texts(discharge_df)
    r_texts, r_hadm = get_texts(radiology_df)

    # ── Stage 3A: TF-IDF + TruncatedSVD ─────────────────────────
    # Paper: separate 500-term vocab for discharge and radiology;
    #        SVD retains 80% variance (136 and 198 components).
    print("\n" + "=" * 60)
    print("STAGE 3A — TF-IDF + TruncatedSVD")
    print("=" * 60)

    disch_tfidf_df, n_disch_svd = tfidf_svd_pipeline(
        d_texts, d_hadm, prefix="disch")
    radio_tfidf_df, n_radio_svd = tfidf_svd_pipeline(
        r_texts, r_hadm, prefix="radio")

    print(f"\n  SVD components — Discharge: {n_disch_svd} | Radiology: {n_radio_svd}")
    print(f"  (paper reports 136 discharge, 198 radiology)")

    # ── Stage 3B: BioBERT + PCA ───────────────────────────────────
    # Paper: mean-pool BioBERT hidden states → 768-dim vector;
    #        PCA retains 90% variance (113 and 115 components).
    print("\n" + "=" * 60)
    print("STAGE 3B — BioBERT + PCA")
    print("=" * 60)

    all_bert_cached = all(
        os.path.exists(os.path.join(CACHE_DIR, f))
        for f in ["disch_bert_pca.npy", "disch_bert_pca_meta.npz",
                  "radio_bert_pca.npy", "radio_bert_pca_meta.npz"]
    )
    if all_bert_cached:
        print("\n  All BioBERT+PCA outputs cached — skipping model load.")
        tokenizer  = None
        bert_model = None
    else:
        print(f"\n  Loading BioBERT : {BERT_MODEL_NAME}")
        tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
        bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME)
        print("  BioBERT loaded.")

    disch_bert_df, n_disch_pca = bert_pca_pipeline(
        d_texts, d_hadm, prefix="disch",
        tokenizer=tokenizer, bert_model=bert_model,
        device=device, bert_batch_size=bert_batch_size)
    radio_bert_df, n_radio_pca = bert_pca_pipeline(
        r_texts, r_hadm, prefix="radio",
        tokenizer=tokenizer, bert_model=bert_model,
        device=device, bert_batch_size=bert_batch_size)

    if bert_model is not None:
        del bert_model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("\n  BioBERT freed from memory.")

    print(f"\n  PCA components — Discharge: {n_disch_pca} | Radiology: {n_radio_pca}")
    print(f"  (paper reports 113 discharge, 115 radiology)")

    # ── Stage 4: Assemble + save ──────────────────────────────────
    text_path = os.path.join(OUTPUT_DIR, "text_only_dataset.csv")

    if os.path.exists(text_path):
        print(f"\n  [CACHE HIT] Loading text_only_dataset.csv ...")
        text_df = pd.read_csv(text_path, low_memory=False)
        print(f"  Loaded shape : {text_df.shape}")
    else:
        text_df = assemble_text_features(
            labels, disch_tfidf_df, radio_tfidf_df,
            disch_bert_df, radio_bert_df)
        text_df.to_csv(text_path, index=False)
        print(f"\n  ✅  Saved : {text_path}")

    # ── Summary ───────────────────────────────────────────────────
    tfidf_cols = [c for c in text_df.columns if "tfidf_svd" in c]
    bert_cols  = [c for c in text_df.columns if "bert_pca"  in c]

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)
    print(f"  Patients (rows)           : {len(text_df):,}")
    print(f"  Total feature columns     : {text_df.shape[1]}")
    print(f"  Discharge TF-IDF SVD cols : {sum(1 for c in tfidf_cols if c.startswith('disch'))}")
    print(f"  Radiology  TF-IDF SVD cols: {sum(1 for c in tfidf_cols if c.startswith('radio'))}")
    print(f"  Discharge BioBERT PCA cols: {sum(1 for c in bert_cols  if c.startswith('disch'))}")
    print(f"  Radiology  BioBERT PCA cols: {sum(1 for c in bert_cols if c.startswith('radio'))}")
    if "disch_note_present" in text_df.columns:
        print(f"  Discharge note coverage   : {text_df['disch_note_present'].mean()*100:.1f}%")
    if "radio_note_present" in text_df.columns:
        print(f"  Radiology note coverage   : {text_df['radio_note_present'].mean()*100:.1f}%")
    print(f"  Output saved to           : {text_path}")
    print(f"  Total runtime             : {(time.time()-pipeline_start)/60:.1f} min")

    return text_df


text_df = main()
text_df.head()



TEXT-ONLY FEATURE EXTRACTION PIPELINE
Replicating paper Sections 6.1 – 6.4
  ⚠️   No GPU — running on CPU
      BERT batch size : 8 (CPU-safe)
      Estimated BioBERT time : 4–8 hours
      TIP: Use Google Colab (free GPU) for 50× speedup.
LOADING LABELS


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\anura\\OneDrive\\Desktop\\data\\labels.csv'